# 03 · Viabilidad de años (2010–2025)

**Objetivo:** revisar, fuente por fuente, qué años existen sobre la provincia de Cusco, para confirmar la ventana de **15 periodos (2010 a 2025)**.

- Solo lee datos y no escribe archivos.
- Landsat tiene el mismo nombre por sensor, así que se cuenta por sensor y por año.
- GEOBOSQUES se revisa con sus imágenes locales, porque su API nacional todavía no tiene código.

**Estado:** notebook nuevo, todavía sin ejecutar.


## 1. Configuración


In [ ]:
from datetime import date, datetime
from pathlib import Path
import hashlib
import io
import json
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
ANIO_INI, ANIO_FIN = 2010, 2025  # 15 periodos anuales: 2010 -> 2025

# Bandas por sensor: nombre común -> banda original
BANDAS_LANDSAT = {
    "L5": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L7": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L8": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
    "L9": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
}
BANDAS_S2 = {"blue": "B2", "green": "B3", "red": "B4", "nir": "B8", "swir1": "B11", "swir2": "B12"}
COLECCIONES_LANDSAT = {
    "L5": "LANDSAT/LT05/C02/T1_L2",
    "L7": "LANDSAT/LE07/C02/T1_L2",
    "L8": "LANDSAT/LC08/C02/T1_L2",
    "L9": "LANDSAT/LC09/C02/T1_L2",
}
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


def md5(ruta):
    """Suma MD5 de un archivo, leído por bloques."""
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
print("Provincia:", prov["nombprov"].iloc[0], "| ventana:", ANIO_INI, "-", ANIO_FIN)


## 2. Landsat por sensor y año

Se cuentan las escenas sobre la provincia. Landsat 5 debería tener escenas en 2010, y Landsat 9 empieza en 2021.


In [ ]:
filas_landsat = []
for sensor, cid in COLECCIONES_LANDSAT.items():
    base = ee.ImageCollection(cid).filterBounds(aoi)
    for anio in range(ANIO_INI, ANIO_FIN + 1):
        n = base.filterDate(f"{anio}-01-01", f"{anio + 1}-01-01").size().getInfo()
        filas_landsat.append({"año": anio, "sensor": sensor, "escenas": n})

df_landsat = (pd.DataFrame(filas_landsat)
              .pivot(index="año", columns="sensor", values="escenas")
              .fillna(0).astype(int))
df_landsat["Landsat (total)"] = df_landsat.sum(axis=1)
df_landsat


## 3. Sentinel-2 por año

Sentinel-2 empieza en 2016 y es denso desde 2019.


In [ ]:
base_s2 = ee.ImageCollection(COLECCION_S2).filterBounds(aoi)
conteo_s2 = {}
for anio in range(ANIO_INI, ANIO_FIN + 1):
    conteo_s2[anio] = base_s2.filterDate(f"{anio}-01-01", f"{anio + 1}-01-01").size().getInfo()

print("Escenas de Sentinel-2 por año:", conteo_s2)


## 4. MapBiomas y WorldCover

MapBiomas tiene una banda por año. WorldCover tiene un producto por cada año (2020 y 2021).


In [ ]:
mb = ee.Image(MB_ASSET)
bandas_mb = mb.bandNames().getInfo()
anios_mb = {int(b.split("_")[1]) for b in bandas_mb if b.startswith("classification_")}
print("MapBiomas con banda dentro de la ventana:", sorted(a for a in anios_mb if ANIO_INI <= a <= ANIO_FIN))

wc_anios = set()
for anio, cid in [(2020, "ESA/WorldCover/v100"), (2021, "ESA/WorldCover/v200")]:
    n = ee.ImageCollection(cid).filterBounds(aoi).size().getInfo()
    if n > 0:
        wc_anios.add(anio)
print("WorldCover con datos sobre la provincia:", sorted(wc_anios))


## 5. GEOBOSQUES (imágenes locales)


In [ ]:
carpeta_gb = RAIZ / "data" / "raw" / "GEOBOSQUES" / "anual"
anios_gb = {int(re.search(r"bosque_(\d{4})_", p.name).group(1))
            for p in carpeta_gb.glob("bosque_*_provincia_cusco.png")}
print("GEOBOSQUES, imágenes locales:", sorted(anios_gb))
print("Faltantes entre 2016 y 2025:", [a for a in range(2016, 2026) if a not in anios_gb] or "ninguno")


## 6. Matriz de disponibilidad 2010–2025

✔ = hay datos ese año, — = no hay datos.

La **serie principal** exige Landsat y MapBiomas en el mismo año. Sentinel-2 y WorldCover son complementos.


In [ ]:
anios = list(range(ANIO_INI, ANIO_FIN + 1))
s2_por_anio = pd.Series(conteo_s2).reindex(anios).fillna(0)

matriz = pd.DataFrame(index=pd.Index(anios, name="año"))
matriz["Landsat"] = df_landsat["Landsat (total)"].reindex(anios).fillna(0).gt(0)
matriz["Sentinel-2"] = s2_por_anio.gt(0)
matriz["MapBiomas C4"] = [a in anios_mb for a in anios]
matriz["WorldCover"] = [a in wc_anios for a in anios]
matriz["GEOBOSQUES (imagen)"] = [a in anios_gb for a in anios]
matriz["OpenStreetMap"] = False  # foto de 2026, sin años anteriores
matriz["Serie principal"] = matriz["Landsat"] & matriz["MapBiomas C4"]

matriz.replace({True: "✔", False: "—"})


## 7. Años sin serie principal

Si aparece algún año aquí, hay que revisarlo antes de cerrar la ventana.


In [ ]:
con_serie = [a for a in anios if matriz.loc[a, "Serie principal"]]
sin_serie = [a for a in anios if not matriz.loc[a, "Serie principal"]]
print("Años con serie principal completa:", con_serie)
print("Años sin serie principal:", sin_serie or "ninguno")
